# 統計2級-06. 重回帰分析

複数の説明変数から1つの結果を予測・説明するのが**重回帰**。
2級頻出の、決定係数・偏回帰係数・回帰係数の検定・ダミー変数・多重共線性を学びます。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv('../data/students_scores.csv')
df.head()

## 1. 単回帰の復習 → 重回帰へ

`数学` を `英語`・`国語`・`勉強時間` から説明します。
$$ 数学 = b_0 + b_1\,英語 + b_2\,国語 + b_3\,勉強時間 $$

`statsmodels` を使うと、表計算ソフトのような回帰の出力が得られます。

In [ ]:
model = smf.ols('数学 ~ 英語 + 国語 + 勉強時間', data=df).fit()
print(model.summary())

## 2. 出力の読み方（2級の頻出ポイント）

- **coef（偏回帰係数）**：他の変数を一定にしたとき、その変数が1増えると数学が何点変わるか
- **R-squared（決定係数）**：当てはまりの良さ（0〜1、1に近いほど良い）
- **Adj. R-squared（自由度調整済み）**：変数の数を考慮した決定係数。変数比較に使う
- **P>|t|（回帰係数の検定）**：その変数が本当に効いているか（0.05未満で有意）

In [ ]:
print('決定係数 R^2      :', round(model.rsquared, 3))
print('自由度調整済み R^2:', round(model.rsquared_adj, 3))
print('\n偏回帰係数:')
print(model.params.round(3))
print('\n各係数のp値:')
print(model.pvalues.round(4))

## 3. 予測してみる

In [ ]:
new = pd.DataFrame({'英語':[70], '国語':[75], '勉強時間':[2.0]})
print('予測される数学の点数:', round(model.predict(new)[0], 1))

## 4. ダミー変数（質的データを回帰に入れる）

`クラス`(A/B/C)のような質的変数は、0/1の**ダミー変数**にして投入します。
`C(クラス)` と書くと自動でダミー化されます。

In [ ]:
model2 = smf.ols('数学 ~ 英語 + C(クラス)', data=df).fit()
print(model2.params.round(2))
print('→ [T.B],[T.C] は基準クラスAと比べた差を表す')

## 5. 多重共線性（マルチコ）

説明変数どうしが強く相関していると、係数が不安定になる問題。
**VIF**（分散拡大係数）で確認します。VIFが10を超えると要注意。

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = df[['英語', '国語', '勉強時間']].copy()
X['const'] = 1
vif = pd.Series(
    [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    index=X.columns)
print(vif.round(2))
print('→ const以外が10未満なら多重共線性は問題なし')

---
## 🏆 練習問題

**問1.** `英語` を `数学`・`国語`・`勉強時間` で説明する重回帰を作り、
決定係数と、有意な（p<0.05）説明変数を答えよう。

**問2.** `sales_btob.csv` で `商談金額` を `業界`（ダミー）で説明する回帰を作り、
どの業界が金額を押し上げ／押し下げているか読み取ろう。

**問3.** 問1のモデルに`クラス`のダミーを加えると自由度調整済みR²は上がる？下がる？確かめよう。

In [ ]:
# 問1


In [ ]:
# 問2
btob = pd.read_csv('../data/sales_btob.csv')


<details><summary>解答例</summary>

```python
m = smf.ols('英語 ~ 数学 + 国語 + 勉強時間', data=df).fit()
print(m.rsquared, m.pvalues)
smf.ols('商談金額 ~ C(業界)', data=btob).fit().params
```
</details>